In [ ]:
import cv2
from pathlib import Path
from PIL import Image
import pandas as pd
import sys

import numpy as np
from sklearn.model_selection import train_test_split
import albumentations as A
import os

In [3]:
sys.path.append('../..')

%load_ext autoreload
%autoreload 2

In [4]:
def filter_dataset(
        df,
        path_to_dataset,
        class_names_to_remove=["yeezy_slide"],
        bad_images_md_path="../2-exploration/bad_images.md"
):
    """

    """
    # Вычищаем Yeezy Slide, т.к. они не укладываются в наши представления о кроссовках
    df = df.query('sneaker_class != "yeezy_slide"')
    # Фильтруем битые картинки, рисунки и т.п.
    # Плохие картинки мы отсмотрели вручную. Список можно увидеть в файле bad_images
    with open(bad_images_md_path) as f:
        bad_images = f.read().strip().split("\n")

    bad_image_paths = [
        image[image.find("(") + 1 : -1] for image in bad_images if len(image) > 0
    ]  # В md указаны пути в формате ![name](path). Поэтому путь это все, что находится в круглых скобках ()

    bad_image_paths = [
        os.path.relpath(
            image, start=path_to_dataset
        )  # Тут заменил, надо заменить пути в bad_images.md
        for image in bad_image_paths
    ]

    print(f"Bad images len: {len(bad_image_paths)}")

    df = df[df["path"].apply(lambda x: x not in bad_image_paths)]
    
    print(f"Dataframe size: {df.shape[0]}")

    return df

In [5]:
# Изображения хранятся в директории так, что каждой модели кроссовок соответствует
# своя папка. Чтобы можно было рассчитывать статистики, мы собираем в датафрейм относительный
# путь до каждого из изображений, и рассчитываем агрегаты на основе этого датафрейма
from src.data.utils.eda_utils import directory_to_dataframe

path_to_dataset = Path("../../data/01_raw/sneakers-dataset")
df = directory_to_dataframe(path_to_dataset)
# Вычищаем Yeezy Slide, т.к. они не укладываются в наши представления о кроссовках
df = filter_dataset(
    df,
    path_to_dataset
)

df.head()

Bad images len: 12
Dataframe size: 5796


,path,sneaker_class
0,reebok_classic_leather/0071.jpg,reebok_classic_leather
1,reebok_classic_leather/0065.jpg,reebok_classic_leather
2,reebok_classic_leather/0059.jpg,reebok_classic_leather
3,reebok_classic_leather/0058.jpg,reebok_classic_leather
4,reebok_classic_leather/0064.jpg,reebok_classic_leather


In [6]:
path, y = df['path'], df['sneaker_class']

In [7]:
# Делим на train+val и test, а потом train и val
path_train_val, path_test, y_train_val, y_test = train_test_split(
    path, y, test_size=0.2, stratify=y, random_state=42
)
path_train, path_val, y_train, y_val = train_test_split(
    path_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=42
)  # 0.25*0.8=0.2

In [17]:
train_df = pd.DataFrame({'path': path_train_val, 'sneaker_class': y_train_val})
train_df.to_csv('../../data/01_raw/train_images.csv', index=False)

In [18]:
test_df = pd.DataFrame({'path': path_test, 'sneaker_class': y_test})
test_df.to_csv('../../data/01_raw/test_images.csv', index=False)

In [8]:
# Пайплайн аугментаций Albumentations
augmenter = A.Compose([
    A.RandomBrightnessContrast(p=0.5),
    A.GaussianBlur(p=0.2),
    A.MotionBlur(p=0.2),
    A.OneOf([
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.OpticalDistortion(p=0.3),
        A.GridDistortion(p=0.3),
    ], p=0.7),
    A.CoarseDropout(max_holes=1, max_height=16, max_width=16, p=0.3)
])

/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/var/folders/t1/3rmgtp792xl3ffv_d2f7y7v40000gn/T/ipykernel_45867/2785960658.py:11: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=1, max_height=16, max_width=16, p=0.3)


In [10]:
# Функция чтения и аугментации изображений
def load_image(img_path): #, augmenter):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # градации серого для классических признаков
    if img is None:
        raise FileNotFoundError(f"Image not found: {img_path}")
    # augmented = augmenter(image=img)
    # return augmented['image']
    return img


In [11]:
# Загрузка данных из папок с классами и извлечение признаков
def augment_dataset(
    base_path,
    augment_path,
    images_path,
    augmenter,
    augment_cnt
):

    for img_file in images_path:
        try:
            img = load_image(base_path / img_file)

            if not (augment_path / img_file).parent.exists():
                os.makedirs((augment_path / img_file).parent)

            cv2.imwrite(augment_path / img_file, img)

            for cnt in range(augment_cnt):

                augmented_image = augmenter(image=img)['image']
                aug_output_path = str(Path(img_file).with_suffix("")) + f'_{cnt}' + str(Path(img_file).suffix)

                if not (augment_path / aug_output_path).parent.exists():
                    os.makedirs((augment_path / aug_output_path).parent)

                cv2.imwrite(augment_path / aug_output_path, augmented_image)

        except Exception as e:
            print(f"Skipping image {base_path / img_file} due to error: {e}")


In [12]:
augment_path = Path("../../data/02_intermediate/sneakers-dataset")

if not augment_path.exists():
    os.makedirs(augment_path)

In [13]:
augment_dataset(
    path_to_dataset,
    augment_path,
    path_train.to_list(),
    augmenter,
    augment_cnt=5
)